In [ ]:
# Load in required libraries for analysis
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)

In [ ]:
PERMANOVA_repeat_measures <- function(
    D,
    subject, subject_data = NULL,
    sample_data = NULL,
    metadata_order = c(names(subject_data), names(sample_data)),
    permutations = 999, ncores = 1,
    by_type = "terms") {
  
  if (!inherits(D, "dist")) {
    stop("D must be a dist object")
  }
  
  if (!is.character(subject)) stop("subject must be a character vector")
  if (length(subject) != nrow(as.matrix(D))) {
    stop("Length of subject must match number of samples in D")
  }
  if (!identical(names(subject), rownames(as.matrix(D)))) {
    stop("Names of subject vector must match sample names in D")
  }
  
  if (is.null(subject_data) & is.null(sample_data)) {
    stop("At least one of subject_data or sample_data must be provided")
  }
  
  if (is.null(subject_data)) {
    subject_data <- data.frame(.placeholder = 1, row.names = unique(subject))
  }
  
  if (length(unique(subject)) != nrow(subject_data)) {
    stop("Number of unique subjects must match rows in subject_data")
  }
  if (!setequal(unique(subject), rownames(subject_data))) {
    stop("Row names of subject_data must match unique values in subject")
  }
  
  if (is.null(sample_data)) {
    sample_data <- data.frame(.placeholder = 1, row.names = rownames(as.matrix(D)))
  }
  if (nrow(as.matrix(D)) != nrow(sample_data)) {
    stop("sample_data must have a row for each sample in D")
  }
  if (!identical(rownames(as.matrix(D)), rownames(sample_data))) {
    stop("Row names of sample_data must match row names of D")
  }
  
  if (length(intersect(names(subject_data), names(sample_data))) > 0) {
    stop("Column names must not overlap between subject_data and sample_data")
  }
  
  if (!all(metadata_order %in% c(names(subject_data), names(sample_data)))) {
    stop("metadata_order must only include columns from subject_data or sample_data")
  }
  
  mtdat <- cbind(subject_data[subject, , drop = FALSE], sample_data)
  
  if (any(is.na(mtdat[, metadata_order, drop = FALSE]))) {
    stop("Missing values found in model metadata")
  }
  
  # Construct formula explicitly
  formula <- as.formula(paste("D ~", paste(metadata_order, collapse = " + ")))
  ad <- vegan::adonis2(formula, permutations = 0, data = mtdat[, metadata_order, drop = FALSE], by = by_type)
  
  R2 <- ad$R2
  names(R2) <- rownames(ad)
  
  library(permute)
  doParallel::registerDoParallel(ncores)
  
  nullsamples <- foreach::`%dopar%`(
    foreach::foreach(i = seq_len(permutations), .combine = cbind),
    {
      subject.i <- sample(unique(subject))
      sample.i <- shuffle(nrow(sample_data), control = how(blocks = subject))
      
      i.subject_data <- subject_data[subject.i, , drop = FALSE]
      rownames(i.subject_data) <- rownames(subject_data)
      
      mtdat_perm <- cbind(i.subject_data[subject, , drop = FALSE],
                          sample_data[sample.i, , drop = FALSE])
      perm_ad <- vegan::adonis2(formula, permutations = 0, data = mtdat_perm[, metadata_order, drop = FALSE], by = by_type)
      r2_perm <- perm_ad$R2
      matrix(r2_perm, ncol = 1, dimnames = list(names(r2_perm), NULL))
    })
  
  doParallel::stopImplicitCluster()
  
  n <- length(R2)
  R2[n - 1] <- 1 - R2[n - 1]
  nullsamples[n - 1, ] <- 1 - nullsamples[n - 1, ]
  
  exceedances <- rowSums(nullsamples > R2)
  P <- (exceedances + 1) / (permutations + 1)
  P[n] <- NA
  
  ad$`Pr(>F)` <- P
  return(ad)
}

In [ ]:
metadata <- read.csv("DOME-16S-Nasal-Metadata.csv")

metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

metadata<- metadata %>%
  # Cohort based on Sample.ID prefix
  mutate(Cohort = case_when(
    startsWith(Sample.ID, "C") ~ "Cow",
    startsWith(Sample.ID, "D") ~ "Non-Farmer",
    startsWith(Sample.ID, "W") ~ "Farmer",
    TRUE ~ "Unknown"
  ))

In [ ]:
GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv"))
relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  # prevent divide by zero
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")

# Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))


In [ ]:
# Prepare abundance matrix
abund_matrix <- relative_abundance_genus
rownames(abund_matrix) <- abund_matrix$Sample.ID
abund_matrix$Sample.ID <- NULL
abund_matrix <- abund_matrix[rowSums(abund_matrix) > 0, ]

# Filter metadata to match abundance matrix
nasal_metadata <- metadata[metadata$Sample.ID %in% rownames(abund_matrix), ]

# Create unique Sample_Site_Time identifier
nasal_metadata <- nasal_metadata %>%
  mutate(
    Sample_Site_Time = paste(Subject, Site, Year, Season, sep = "-"),
    Sample_Site_Time = make.unique(Sample_Site_Time)
  )
rownames(nasal_metadata) <- nasal_metadata$Sample_Site_Time

# Align abundance matrix to metadata
abund_matrix <- abund_matrix[nasal_metadata$Sample.ID, , drop = FALSE]
rownames(abund_matrix) <- nasal_metadata$Sample_Site_Time

# Define repeated-measures subjects (Subject + Site)
subjects <- paste(nasal_metadata$Subject, nasal_metadata$Site, sep = "_")
names(subjects) <- rownames(nasal_metadata)

# Subject-level metadata
subject_data <- nasal_metadata %>%
  select(Subject, Site, Cohort) %>%
  distinct()
rownames(subject_data) <- paste(subject_data$Subject, subject_data$Site, sep = "_")
subject_data <- subject_data %>% select(Cohort)

# Sample-level metadata
sample_data <- nasal_metadata %>%
  select(Site, Year, Season)

In [ ]:
write.csv(sample_data, "PERMANOVA-Sample-Data.csv")
write.csv(abund_matrix, "PERMANOVA-Abundance-Matrix.csv")
write.csv(nasal_metadata, "PERMANOVA-Nasal-Metadata.csv")
write.csv(subject_data, "PERMANOVA-Subject-Data.csv")

In [ ]:
bray_dist <- vegdist(abund_matrix, method = "bray")
write.csv(bray_dist, "PERMANOVA-Bray-Curtis-Data.csv")

In [ ]:
result <- PERMANOVA_repeat_measures(
  D = as.dist(bray_dist),
  subject = subjects,
  subject_data = subject_data,
  sample_data = sample_data,
  metadata_order = c("Cohort", "Site", "Year", "Season"),
  permutations = 999,
  ncores = 16,
  by_type = "terms"
)
# Extract p-values
pvals <- result$`Pr(>F)`

# Apply BH correction
p_adj <- p.adjust(pvals, method = "BH")

# Add to result table
result$`Pr(>F)_BH` <- p_adj

write.csv(result, "PERMANOVA-Specified-Variables-Cohort-Site-Year-Season.csv")

In [ ]:

result <- PERMANOVA_repeat_measures(
  D = as.dist(bray_dist),
  subject = subjects,
  subject_data = subject_data,
  sample_data = sample_data,
  metadata_order = c("Site"),
  permutations = 999,
  ncores = 8,
  by_type = "terms"
)

all_variables_PERMANOVA <- PERMANOVA_repeat_measures(
  D = as.dist(bray_dist),
  subject = subjects,
  subject_data = subject_data,
  sample_data = sample_data,
  # metadata_order = c("Site", "Year", "Season"),
  permutations = 999,
  ncores = 16,
  by_type = "terms"
)
write.csv(all_variables_PERMANOVA, "PERMANOVA-Sample-Data-Site-Year-Season.csv", row.names = TRUE)